# Tag ingestion & model download

Minimal preparation notebook — run this **once** before the main pipeline and `variance_fast.ipynb`.

1. Load raw tag CSV → standard DataTree
2. Download CMEMS model over the tag's spatial/temporal domain
3. Save `model.zarr` to object storage


In [ ]:
import pandas as pd
import sys

sys.path.append("../pangeo-fish-public")

# ── Configuration ─────────────────────────────────────────────────────────────
TAG_TYPE = "lotek"
tag_name = "281B-4949"
RAW_DATA_PATH = "/path/to/281B-4949_Basic Log_1.csv"
TAGGING_EVENTS_PATH = "/path/to/281B-4949_tagging_events.csv"

scratch_root = "s3://gfts-ifremer/tuna/run/capetienne/generic"
storage_options = {
    "anon": False,
    "profile": "gfts",
    "client_kwargs": {
        "endpoint_url": "https://s3.gra.perf.cloud.ovh.net",
        "region_name": "gra",
    },
}
target_root = f"{scratch_root}/{tag_name}_larger_bbox1"

TAG_ROOT = "/path/to/tag_data_root"  # local folder for {tag_name}/ subfolder

bbox = {"latitude": [34.0, 48.0], "longitude": [-6.0, 20.0], "max_depth": 450}

## 1. Load tag

In [ ]:
from pangeo_fish.helpers import load_tag

tag, tag_log, time_slice = load_tag(
    tag_root=TAG_ROOT,
    tag_name=tag_name,
)
print(f"Deployment: {time_slice.start} → {time_slice.stop}")
tag

## 2. Download model from Copernicus Marine

In [ ]:
import copernicusmarine
import xarray as xr
from pangeo_fish.io import prepare_dataset

ds_thetao_zos = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    variables=["thetao", "zos"],
    minimum_longitude=bbox["longitude"][0],
    maximum_longitude=bbox["longitude"][1],
    minimum_latitude=bbox["latitude"][0],
    maximum_latitude=bbox["latitude"][1],
    start_datetime=time_slice.start.strftime("%Y-%m-%d"),
    end_datetime=time_slice.stop.strftime("%Y-%m-%d"),
)

static_var = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_static",
    variables=["deptho", "mask"],
    minimum_longitude=bbox["longitude"][0],
    maximum_longitude=bbox["longitude"][1],
    minimum_latitude=bbox["latitude"][0],
    maximum_latitude=bbox["latitude"][1],
)
static_var = static_var.assign_coords(longitude=ds_thetao_zos["longitude"].values)

ds_all = xr.merge([ds_thetao_zos, static_var], compat="no_conflicts")
ds_all = ds_all.rename({"latitude": "lat", "longitude": "lon"})
model = prepare_dataset(ds_all)
model

## 3. Save model to zarr

In [ ]:
import dask

model_zarr = f"{target_root}/model.zarr"

with dask.config.set(scheduler="threads"):
    model.chunk({"time": 1, "depth": 20, "lat": 100, "lon": 100}).to_zarr(
        model_zarr,
        mode="w",
        storage_options=storage_options,
        zarr_version=2,
    )

print(f"Saved → {model_zarr}")
del model

## 4. Verify

In [ ]:
import xarray as xr

model = xr.open_dataset(
    f"{target_root}/model.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
model